# 05. Baseline & Machine Learning Model Training

## Methodological Framing
We evaluate models against a transparent heuristic baseline using a primary chronological train/validation/test split. The operational problem is future deterioration; therefore, temporal ordering is strictly preserved. Client overlap across temporal splits is explicitly allowed.

### Models:
1. **Heuristic Baseline:** Hand-crafted rule weighting historical clicks and position decay.
2. **Logistic Regression:** Linear interpretable classification benchmark.
3. **Random Forest:** Non-linear tree ensemble with balanced subsample weighting.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

root_dir = Path.cwd().parent if Path.cwd().name == "work" else Path.cwd()
sys.path.insert(0, str(root_dir))

from src.config import load_config
from src.data import load_dataset
from src.features import build_feature_table
from src.labels import construct_operational_label
from src.splits import create_chronological_split
from src.baseline import HeuristicRanker
from src.model import ModelPipeline

config = load_config()

## Step 1: Chronological Partitioning & Model Fitting

In [ ]:
try:
    df = load_dataset(config.raw_data_path, config)
    is_eligible, y_target, diag = construct_operational_label(df, config.label)
    
    df_el = df.loc[is_eligible].copy()
    y_el = y_target.loc[is_eligible].copy()
    X, f_names = build_feature_table(df_el, config.schema_mapping.get("features", []))
    
    split = create_chronological_split(df_el, config=config.validation)
    print(f"Temporal Split - Train: {len(split.train_indices):,}, Val: {len(split.val_indices):,}, Test: {len(split.test_indices):,}")
    
    # Train Baseline
    baseline = HeuristicRanker()
    baseline.fit(X.iloc[split.train_indices], y_el.iloc[split.train_indices])
    
    # Train Models on Train set only (No test tuning)
    rf = ModelPipeline(model_type="random_forest", random_seed=config.random_seed)
    rf.fit(X.iloc[split.train_indices], y_el.iloc[split.train_indices])
    
    lr = ModelPipeline(model_type="logistic_regression", random_seed=config.random_seed)
    lr.fit(X.iloc[split.train_indices], y_el.iloc[split.train_indices])
    
    print("Models fitted successfully without leakage.")
except FileNotFoundError:
    print("[STATUS: AWAITING REAL DATA EXECUTION] - Connect warehouse data to execute model fitting.")